In [2]:
# Standard library imports
import os
import time
import datetime
from collections import defaultdict

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pytz
import joblib
import ta
from fpdf import FPDF
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    f1_score,
    accuracy_score
)
from sklearn.utils import class_weight
import MetaTrader5 as mt5

# TensorFlow / Keras imports
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras.callbacks import EarlyStopping

# Local application imports
from functions.balanced import *
from functions.report import *

from sklearn.ensemble import RandomForestClassifier

In [5]:
data = pd.read_csv('training_data_meta/eurusd_dataset_2025_test.csv')
data = data.drop(['DateTime'], axis=1)

X = data.drop(['label'], axis=1)
y = data['label'].astype(int)

# split_index = int(len(data) * 0.8)
# train_data = data.iloc[:split_index]
# test_data = data.iloc[split_index:]

# xtrain = train_data.drop(['label'], axis=1)
# ytrain = test_data['label'].astype(int)

# xtest = test_data.drop(['label'], axis=1)
# ytest = test_data['label'].astype(int)

Only start trading at 08:10 and stop opening trades at 21

In [24]:
# Initialize
X_full = X.copy()
X_full['pred'] = np.nan
X_full['trade_pnl'] = 0.0
X_full['lot_size'] = 0.0
X_full['trade_open'] = False
X_full['balance'] = np.nan

# Parameters
sequence_length = 12
window_size = 20000
val_size = 5000
step = 5000
cost_per_trade = 3.26 # review this, it can be changed to be in euros    
pip_value_per_standard_lot = 10
initial_account_balance = 10000.0
risk_per_trade_percentage = 0.01
steps = 12 # ver se isto influencia, meti todas as funções dependentes desta variável
extra_steps = 0
class_to_direction = {0: 0, 1: 1}
threshold = 0.7

# Tracking
current_account_balance = initial_account_balance
f1_per_window, profit_per_window, trade_per_window, window_indices = [], [], [], []
vol_per_window, local_vol_per_window, losing_profit, winning_profit, reports = [], [], [], [], []
train_distributions = []
val_distributions = []
epoch_counts = []
reports_1 = []

profit_per_class = defaultdict(float)
trades_per_class = defaultdict(int)
winning_trades, losing_trades = 0, 0

# to remove bad trades
X_full['flag'] = X_full['hour'].isin([8, 21]) # find a way to flag the 8h10 trades also
columns_to_drop = ['pred', 'trade_pnl', 'lot_size', 'trade_open', 'balance', 'flag']

for w, start in enumerate(range(0, len(X_full) - window_size - val_size - steps, step)):
    print(f"\n Window {w}")

    # Split
    train_X_slice = X_full.iloc[start : start + window_size]
    train_y = y[start : start + window_size]

    val_X_slice = X_full.iloc[start + window_size : start + window_size + val_size]
    val_y = y[start + window_size : start + window_size + val_size]

    train_flag = train_X_slice['flag']
    val_flag = val_X_slice['flag']

    train_X = train_X_slice.drop(columns=columns_to_drop)
    val_X = val_X_slice.drop(columns=columns_to_drop)

    # Standardize
    scaler = MinMaxScaler()
    train_X_scaled = scaler.fit_transform(train_X)
    val_X_scaled = scaler.fit_transform(val_X)

    train_X_seq, train_y_seq = create_lstm_sequences(train_X_scaled, train_y.values, sequence_length)
    val_X_seq, val_y_seq = create_lstm_sequences(val_X_scaled, val_y.values, sequence_length)

    train_flag_targets = train_flag[sequence_length:]
    val_flag_targets = val_flag[sequence_length:]
    # Filter out sequences where target point was flagged
    train_mask = train_flag_targets == 0
    val_mask = val_flag_targets == 0

    train_X_seq = train_X_seq[train_mask]
    train_y_seq = train_y_seq[train_mask]

    '''train_y_cat = to_categorical(train_y_seq, num_classes=3)
    val_y_cat = to_categorical(val_y_seq, num_classes=3)'''
    train_y_cat = train_y_seq.copy()
    val_y_cat = val_y_seq.copy()

    # Build & train model
    input_features = train_X_seq.shape[2]
    sequence_length = train_X_seq.shape[1]
    model = build_model_rnn(sequence_length, input_features)

    cw = dict(enumerate(class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(train_y_seq), y=train_y_seq)))
    # cw = {0: 1, 1: 100} # fine tune this
    # early_stop = EarlyStopping(
    #     monitor='val_loss',  # or 'val_accuracy'
    #     patience=5,          # wait 5 epochs for improvement
    #     restore_best_weights=True
    # )

    history = model.fit(train_X_seq, train_y_cat,
            epochs=8, batch_size=32,  class_weight=cw,
            verbose=0, shuffle=False) #, callbacks=[early_stop])
    
    # Store how many epochs were used
    # used_epochs = len(history.history['loss'])
    # epoch_counts.append(used_epochs)

    # Predict
    #Prevê a melhor class e guarda a confiança
    preds_probs = model.predict(val_X_seq, verbose=0)
    # preds = np.argmax(preds_probs, axis=1)
    # confidences = np.max(preds_probs, axis=1)
    preds = (preds_probs.flatten() >= threshold).astype(int)
    confidences = preds_probs.flatten()

    # Set prediction to 0 if confidence < 0.7
    # preds = np.where(confidences >= threshold, preds, 0)

    val_start = start + window_size
    adjusted_start = val_start + sequence_length
    max_len = min(len(preds), len(X_full) - adjusted_start)
    pred_indices = range(adjusted_start, adjusted_start + len(preds))

    #armazena na df
    X_full.loc[pred_indices, 'pred'] = preds[:max_len]
    X_full.loc[pred_indices, 'confidence'] = confidences[:max_len]

    f1_preds = preds[val_mask]
    f1_val = val_y_seq[val_mask]

    unique, counts = np.unique(train_y_seq, return_counts=True)
    train_distribution = dict(zip(unique, counts))

    unique_val, counts_val = np.unique(val_y_cat, return_counts=True)
    val_distribution = dict(zip(unique_val, counts_val))

    sl_tp_map = {
            0: {'sl': None, 'tp': None},
            1: {'sl': 15, 'tp': 15}
        }
      
    avg_duration_by_class = {
            0: None,
            1: 12
        }           

    # Trade Simulation
    min_broker_lot_size = 0.01
    max_broker_lot_size = 50.0
    profit = 0.0
    trades = 0
    last_trade_close_idx = -1

    for i, t in enumerate(pred_indices):

        flag = X_full.at[t, 'flag']
        if flag == 1:
            continue

        if np.isnan(X_full.at[t, 'pred']):
            continue
        if t < last_trade_close_idx:
            continue

        pred = int(X_full.at[t, 'pred'])
        direction = class_to_direction.get(pred, 0)
        if direction == 0: # or direction == -1:
            continue

        sltp = sl_tp_map.get(pred, {'sl': None, 'tp': None})
        if sltp['sl'] is None or sltp['tp'] is None or sltp['sl'] <= 0:
            continue  

        monetary_risk = current_account_balance * risk_per_trade_percentage
        lot_size = monetary_risk / (sltp['sl'] * pip_value_per_standard_lot)
        lot_size = round(min(max(lot_size, min_broker_lot_size), max_broker_lot_size), 2)
        if lot_size < 0.01:
            continue
        
        entry = X_full.at[t + 1, 'Open']
        highs_seq = X_full['High'].iloc[t + 1 : t + 1 + steps].values
        lows_seq = X_full['Low'].iloc[t + 1 : t + 1 + steps].values
        limit = avg_duration_by_class.get(pred, 0) + extra_steps

        result_pips, rel_exit_idx = simulate_trade(entry, highs_seq[:limit], lows_seq[:limit], direction, sltp['sl'], sltp['tp'])
        result_pips -= cost_per_trade

        trade_profit = result_pips * pip_value_per_standard_lot * lot_size
        profit += trade_profit
        trades += 1
        current_account_balance += trade_profit
        last_trade_close_idx = t + rel_exit_idx

        X_full.at[t, 'trade_open'] = True
        X_full.at[t, 'trade_pnl'] = trade_profit
        X_full.at[t, 'lot_size'] = lot_size
        X_full.at[t, 'balance'] = current_account_balance

        profit_per_class[pred] += trade_profit
        trades_per_class[pred] += 1
        if result_pips > 0:
            winning_trades += 1
            winning_profit.append(trade_profit)
        elif result_pips < 0:
            losing_trades += 1
            losing_profit.append(trade_profit)

    # Track metrics
    vol = val_X['Close'].pct_change()
    vol_std = vol.std()
    vol_local = val_X['vol_local'].mean()

    report = classification_report(f1_val, f1_preds, digits=3)
    f1 = f1_score(f1_val, f1_preds, average='weighted')
    f1_per_window.append(f1)
    profit_per_window.append(profit)
    trade_per_window.append(trades)
    window_indices.append(w)
    vol_per_window.append(vol_std)
    local_vol_per_window.append(vol_local)
    reports.append(report)
    train_distributions.append(train_distribution)
    val_distributions.append(val_distribution)

    '''# For training data (train set, not validation)
    train_start = start
    adjusted_train_start = train_start + sequence_length
    max_train_len = len(train_y_seq)
    train_pred_indices = range(adjusted_train_start, adjusted_train_start + max_train_len)
    
    preds_probs_1 = model.predict(train_X_seq, verbose=0)
    preds_1 = np.argmax(preds_probs_1, axis=1)
    confidences_1 = np.max(preds_probs_1, axis=1)

    f1_1 = f1_score(train_y.iloc[:max_train_len], preds_1[:max_train_len], average='weighted')
    report_1 = classification_report(train_y.iloc[:max_len], preds_1[:max_len], digits=4)

    reports_1.append(report_1)
'''
    print(f"Profit: {profit:.2f}, Trades: {trades}, F1: {f1:.3f}, Balance: {current_account_balance:.2f}")


 Window 0
Profit: 1080.53, Trades: 110, F1: 0.939, Balance: 11080.53

 Window 1
Profit: 5538.89, Trades: 66, F1: 0.933, Balance: 16619.42

 Window 2
Profit: 1771.77, Trades: 111, F1: 0.921, Balance: 18391.19

 Window 3
Profit: 3165.89, Trades: 55, F1: 0.951, Balance: 21557.08

 Window 4
Profit: 487.20, Trades: 60, F1: 0.937, Balance: 22044.28

 Window 5
Profit: -5660.63, Trades: 265, F1: 0.591, Balance: 16383.65

 Window 6
Profit: 193.72, Trades: 6, F1: 0.974, Balance: 16577.37

 Window 7
Profit: -1073.05, Trades: 131, F1: 0.853, Balance: 15504.32

 Window 8
Profit: 12.99, Trades: 150, F1: 0.825, Balance: 15517.32

 Window 9
Profit: 2881.71, Trades: 34, F1: 0.975, Balance: 18399.02

 Window 10
Profit: -7595.60, Trades: 324, F1: 0.600, Balance: 10803.42

 Window 11
Profit: 2045.11, Trades: 28, F1: 0.896, Balance: 12848.53

 Window 12


KeyboardInterrupt: 

In [22]:
class_distribution

{np.int64(0): np.int64(16188), np.int64(1): np.int64(1242)}

In [41]:
generate_model_report_pdf(
    steps,
    extra_steps,
    window_indices,
    f1_per_window,
    profit_per_window,
    trade_per_window,
    vol_per_window,
    local_vol_per_window,
    losing_profit,
    winning_profit,
    initial_account_balance,
    window_size,
    val_size,
    step,
    cost_per_trade,
    pip_value_per_standard_lot,
    risk_per_trade_percentage,
    winning_trades,
    losing_trades,
    profit_per_class,
    trades_per_class,
    threshold,
    train_distributions,
    val_distributions,
    report_filename="model_eurusd_both.pdf"
)

C:\Users\marti\AppData\Local\Temp\ipykernel_8208\3126606719.py:177: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", "B", 16)
C:\Users\marti\AppData\Local\Temp\ipykernel_8208\3126606719.py:178: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Trading Model Performance Report", 0, 1, "C")
C:\Users\marti\AppData\Local\Temp\ipykernel_8208\3126606719.py:179: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", "", 10)
C:\Users\marti\AppData\Local\Temp\ipykernel_8208\3126606719.py:180: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}", 0, 1, "C")
C:\Users\


Report generated successfully: model_eurusd_both.pdf


In [32]:
print(reports[7])

              precision    recall  f1-score   support

           0      0.997     0.775     0.872      4238
           1      0.101     0.907     0.181       118

    accuracy                          0.778      4356
   macro avg      0.549     0.841     0.527      4356
weighted avg      0.972     0.778     0.853      4356



In [36]:
train_distributions[0]

{np.int64(0): np.int64(15263), np.int64(1): np.int64(1727)}

In [39]:
val_distributions[0]

{np.int64(0): np.int64(4448), np.int64(1): np.int64(540)}

In [37]:
train_distributions[5]

{np.int64(0): np.int64(16398), np.int64(1): np.int64(1031)}

In [38]:
val_distributions[5]

{np.int64(0): np.int64(4601), np.int64(1): np.int64(387)}